In this code we will only do the ann related things but not any kind of data analysis

In [76]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder

import pandas as pd

In [77]:
df = pd.read_csv('https://raw.githubusercontent.com/JosePadillaMtnz/BreastCancer_KaggleDataset/main/data.csv')

In [78]:
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [79]:
df.drop(columns=['id','Unnamed: 32'], inplace=True)

In [80]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 31 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   diagnosis                569 non-null    str    
 1   radius_mean              569 non-null    float64
 2   texture_mean             569 non-null    float64
 3   perimeter_mean           569 non-null    float64
 4   area_mean                569 non-null    float64
 5   smoothness_mean          569 non-null    float64
 6   compactness_mean         569 non-null    float64
 7   concavity_mean           569 non-null    float64
 8   concave points_mean      569 non-null    float64
 9   symmetry_mean            569 non-null    float64
 10  fractal_dimension_mean   569 non-null    float64
 11  radius_se                569 non-null    float64
 12  texture_se               569 non-null    float64
 13  perimeter_se             569 non-null    float64
 14  area_se                  569 non-null

In [81]:
X = df.drop(columns=['diagnosis']).values
y = df['diagnosis'].values

In [82]:
y.unique()

<StringArray>
['M', 'B']
Length: 2, dtype: str

In [83]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [84]:
# feature scaling is important for neural networks because it helps the model converge faster and can lead to better performance. 
# StandardScaler standardizes the features by removing the mean and scaling to unit variance, which can help the model learn more 
# effectively.
scaler = StandardScaler()
X_train  = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)  


# Label encoding is necessary to convert the categorical labels (in this case, 'M' and 'B') into numerical values 
# that can be used by the neural network.
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

In [85]:
class BreastCancerModel(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()
        )   

    def forward(self, x):
        return self.model(x)



class data_set(Dataset):
    # this is a custom dataset class that inherits from torch.utils.data.Dataset. 
    # it takes in the features (X) and labels (y) as input and converts them into PyTorch tensors.
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.int32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # here we can also add data augmentation or other transformations if needed
        return self.X[idx], self.y[idx]

In [86]:
summary(model, input_size=(X.shape[0], X.shape[1]))
# summary(model, input_size=(1, X.shape[1]))

Layer (type:depth-idx)                   Output Shape              Param #
BreastCancerModel                        [569, 1]                  --
├─Sequential: 1-1                        [569, 1]                  --
│    └─Linear: 2-1                       [569, 16]                 496
│    └─ReLU: 2-2                         [569, 16]                 --
│    └─Linear: 2-3                       [569, 1]                  17
│    └─Sigmoid: 2-4                      [569, 1]                  --
Total params: 513
Trainable params: 513
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 0.29
Input size (MB): 0.07
Forward/backward pass size (MB): 0.08
Params size (MB): 0.00
Estimated Total Size (MB): 0.15

In [87]:
train_dataset = data_set(X_train, y_train)
test_dataset = data_set(X_test, y_test)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=32)

In [88]:
model = BreastCancerModel(num_features=X.shape[1])
loss_fn = nn.BCELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
epochs = 30

In [89]:
for epoch in range(epochs):
    epoch_loss = 0
    for X_batch, y_batch in train_dataloader:

        y_pred = model(X_batch)
        loss = loss_fn(y_pred, y_batch.view(-1,1).float())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step() 

        epoch_loss += loss.item()

    print(f'Epoch {epoch+1}/{epochs} | Average Loss: {epoch_loss/len(train_dataloader)}')


Epoch 1/30 | Average Loss: 0.6403345624605815
Epoch 2/30 | Average Loss: 0.5983118931452434
Epoch 3/30 | Average Loss: 0.5573457956314087
Epoch 4/30 | Average Loss: 0.5259159167607625
Epoch 5/30 | Average Loss: 0.48671600023905437
Epoch 6/30 | Average Loss: 0.4531793077786764
Epoch 7/30 | Average Loss: 0.4248083432515462
Epoch 8/30 | Average Loss: 0.40540852745374045
Epoch 9/30 | Average Loss: 0.37446081240971885
Epoch 10/30 | Average Loss: 0.3491818606853485
Epoch 11/30 | Average Loss: 0.3334619621435801
Epoch 12/30 | Average Loss: 0.3080358445644379
Epoch 13/30 | Average Loss: 0.29214170277118684
Epoch 14/30 | Average Loss: 0.2776055326064428
Epoch 15/30 | Average Loss: 0.2642887403567632
Epoch 16/30 | Average Loss: 0.26047805746396385
Epoch 17/30 | Average Loss: 0.24366973241170248
Epoch 18/30 | Average Loss: 0.22887714803218842
Epoch 19/30 | Average Loss: 0.2190621962149938
Epoch 20/30 | Average Loss: 0.2232406457265218
Epoch 21/30 | Average Loss: 0.2054288923740387
Epoch 22/30 | A

In [92]:
model.eval()
accuracy_list = []
with torch.no_grad():
  for batch_feature, batch_label in  train_dataloader:
    y_pred = model(batch_feature)
    y_pred = (y_pred > 0.8).float()
    accuracy = (y_pred.view(-1) == batch_label).float().mean().item()
    accuracy_list.append(accuracy)


sum(accuracy_list)/len(accuracy_list)

0.9104166666666667

In [91]:
model.eval()
accuracy_list = []
with torch.no_grad():
  for batch_feature, batch_label in  test_dataloader:
    y_pred = model(batch_feature)
    y_pred = (y_pred > 0.8).float()
    accuracy = (y_pred.view(-1) == batch_label).float().mean().item()
    accuracy_list.append(accuracy)


sum(accuracy_list)/len(accuracy_list)

0.8940972238779068